<a href="https://colab.research.google.com/github/taavip/DL4CV/blob/main/HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework #2
## Hands-on object detection and segmentation

This colaboratory contains Homework #2 of the Deep Learning for Computer Vision course, which is due **March 29, Sunday, midnight (23:59 EET time)**. To complete the homework, extract **(File -> Download .ipynb)** and submit to the course webpage.

**NB! Links to your colaboratory will not be accepted as a solution!**

## Submission's rules:

1. Please, submit only .ipynb that you extract from the Colaboratory.
2. Run your homework exercises before submitting (output should be present, preferably restart the kernel and press run all the cells).
3. Do not change the description of tasks in red (even if there is a typo|mistake|etc).
4. Please, make sure to avoid unnecessary long printouts.
5. Each task should be solved right under the question of the task and not elsewhere.
6. Solutions to both regular and bonus exercises should be submitted in one IPYNB file.

Please, steer clear of copying someone else's work. If you discuss assignments with anyone in the course, please, mention their names here:

##List of Homework's exercises:
1. [EX1](#scrollTo=-42AUhyYZlh1&line=1&uniqifier=1) - 8 points
2. [EX2](#scrollTo=1SY9vdZg2_Zi&line=1&uniqifier=1) - 8 points
3. [EX3](#scrollTo=iBfAXUiC3JQ4&line=1&uniqifier=1) - 4 points

# Homework exercise 1 (8 points): Create an object detection model

<font color='red'>**(Homework exercise 1- a)** In this exercise, you will continue working with the dataset you created in HW1. Add bounding boxes annotations to as many images as you think will be enough to get a decent model. Start with a manageable number of bounding boxes per class (e.g., 5) and then add more if model's final performance isn't satisfactory (IoU > 0.6).  We suggest you use [LabelStudio](https://labelstud.io/guide/install) for this annotations. (3 points)</font>

**LabelStudio notes**:
1. After you finished the labeling, download the images and annotations together as an archive.
2. Save annotations in the **Pascal VOC** format.

In [ ]:
from PIL import Image
import numpy as np
import json
import matplotlib
import matplotlib.pyplot as plt
import cv2

In [ ]:
# Upload the .zip archive with images and annotations
from google.colab import files

files.upload()

<font color='red'>Let's visualize one image and its bounding box to understand in what format the bounding box coordinates are stored.

<font color='red'>Read a single image from your dataset and its annotations.

In [ ]:
#### YOUR CODE STARTS HERE ####
import glob, os, xml.etree.ElementTree as ET
# Update this path to where you extracted your Pascal VOC archive
dataset_dir = globals().get('dataset_dir', 'dataset')
# Find images
img_paths = glob.glob(os.path.join(dataset_dir, '**', '*.jpg'), recursive=True) + glob.glob(os.path.join(dataset_dir, '**', '*.png'), recursive=True)
if len(img_paths) == 0:
    raise FileNotFoundError(f'No images found under {dataset_dir}. Update dataset_dir accordingly.')
img_path = img_paths[0]
img = Image.open(img_path).convert('RGB')
# Try to find Pascal VOC XML with same basename
base = os.path.splitext(os.path.basename(img_path))[0]
xml_candidates = glob.glob(os.path.join(dataset_dir, '**', base + '*.xml'), recursive=True)
boxes = []
if xml_candidates:
    tree = ET.parse(xml_candidates[0])
    root = tree.getroot()
    for obj in root.findall('object'):
        cls = obj.find('name').text
        bnd = obj.find('bndbox')
        xmin = int(bnd.find('xmin').text); ymin = int(bnd.find('ymin').text)
        xmax = int(bnd.find('xmax').text); ymax = int(bnd.find('ymax').text)
        width = xmax - xmin; height = ymax - ymin
        top_left_x, top_left_y = xmin, ymin
        boxes.append((cls, top_left_x, top_left_y, width, height))
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(np.array(img))
if boxes:
    cls, top_left_x, top_left_y, width, height = boxes[0]
    draw_rectangle(ax, top_left_x, top_left_y, width, height)
    ax.set_title(f'{os.path.basename(img_path)} — {cls}')
else:
    ax.set_title(f'{os.path.basename(img_path)} — no XML annotation found')
ax.axis('off')
plt.show()
#### YOUR CODE ENDS HERE ####

In [ ]:
def draw_rectangle(ax, top_left_x, top_left_y, width, height, color='red'):
    top_left = (top_left_x, top_left_y)
    bottom_right = (top_left_x + width, top_left_y + height)

    # Plotting each edge of the rectangle on the specified axes
    ax.plot([top_left[0], top_left[0]], [top_left[1], bottom_right[1]], '-', color=color)  # Left edge
    ax.plot([bottom_right[0], bottom_right[0]], [top_left[1], bottom_right[1]], '-', color=color)  # Right edge
    ax.plot([top_left[0], bottom_right[0]], [top_left[1], top_left[1]], '-', color=color)  # Top edge
    ax.plot([top_left[0], bottom_right[0]], [bottom_right[1], bottom_right[1]], '-', color=color)  # Bottom edge

In [ ]:
# Plot the image and the bbox
fig, ax = plt.subplots()
ax.imshow(img)
draw_rectangle(ax, top_left_x, top_left_y, width, height)
plt.show()

<font color='red'>**(Homework exercise 1- b)** Create an object detection dataset class. You can use the FruitDataset from object detection practice as a template. Make sure to convert the coordinates of bounding boxes into the correct format `(class_id, x_center, y_center, width, height)`. (2 points)</font>

In [ ]:
import torch
import torch.nn as nn # all the relevant building blocks
import torch.nn.functional as F # functional interfaces for many operations
from torch.utils.data import Dataset, DataLoader # abstract primitives for handling data in pytorch
from torchvision import transforms

if torch.cuda.is_available():
    print("GPU is available")
    device = torch.device("cuda")
else:
    print("GPU is not available, using CPU instead")
    device = torch.device("cpu")

In [ ]:
#### YOUR CODE STARTS HERE ####
import glob, xml.etree.ElementTree as ET, os
dataset_dir = globals().get('dataset_dir', 'dataset')
xml_paths = glob.glob(os.path.join(dataset_dir, '**', '*.xml'), recursive=True)
classes_set = set()
for xp in xml_paths:
    root = ET.parse(xp).getroot()
    for o in root.findall('object'):
        classes_set.add(o.find('name').text)
classes = sorted(list(classes_set)) if classes_set else ['object']
#### YOUR CODE ENDS HERE ####

In [ ]:
#### YOUR CODE STARTS HERE ####
# Standard sizes used later in the notebook
img_width = 416
img_height = 277

grid_height = 4 # Feel free to change the grid size
grid_width = 5
#### YOUR CODE ENDS HERE ####

cell_width = np.ceil(img_width / grid_width)
cell_height = np.ceil(img_height / grid_height)

K = len(classes)

In [ ]:
class ObjectDetectionDataset(Dataset):
    def __init__(self, dataset_dir, classes, img_width, img_height, grid_height, grid_width):
      super(ObjectDetectionDataset, self).__init__()
      self.dataset_dir = dataset_dir
      self.grid_height = grid_height
      self.grid_width = grid_width
      self.classes = classes
      self.K = len(classes)
      self.img_width = img_width
      self.img_height = img_height
      self.cell_width = np.ceil(self.img_width / self.grid_width)
      self.cell_height = np.ceil(self.img_height / self.grid_height)

      # Load images and annotations (Pascal VOC XMLs)
      import glob, os, xml.etree.ElementTree as ET
      xml_paths = glob.glob(os.path.join(dataset_dir, '**', '*.xml'), recursive=True)
      images = []
      annotations = []
      for xp in xml_paths:
        tree = ET.parse(xp); root = tree.getroot()
        # try to find the image file referenced in XML or by basename
        filename_tag = root.find('filename')
        if filename_tag is not None:
          fname = filename_tag.text
          # try common extensions
          for ext in [fname, fname + '.jpg', fname + '.png']:
            candidate = os.path.join(os.path.dirname(xp), ext)
            if os.path.exists(candidate):
              img_path = candidate
              break
          else:
            img_path = None
        else:
          base = os.path.splitext(os.path.basename(xp))[0]
          candidates = glob.glob(os.path.join(os.path.dirname(xp), base + '.*'))
          img_path = candidates[0] if candidates else None

        if img_path is None:
          continue
        images.append(Image.open(img_path).convert('RGB'))
        ann_list = []
        for obj in root.findall('object'):
          cname = obj.find('name').text
          if cname not in self.classes:
            continue
          class_id = self.classes.index(cname)
          bnd = obj.find('bndbox')
          xmin = float(bnd.find('xmin').text); ymin = float(bnd.find('ymin').text)
          xmax = float(bnd.find('xmax').text); ymax = float(bnd.find('ymax').text)
          w = xmax - xmin; h = ymax - ymin
          cx = xmin + w / 2.0; cy = ymin + h / 2.0
          ann_list.append([class_id, cx, cy, w, h])
        annotations.append(ann_list)

      self.images = images
      self.annotations = annotations
      from torchvision import transforms as T
      self.transform = T.Compose([T.Resize((self.img_height, self.img_width)), T.ToTensor()])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        image = self.transform(image)
        bboxes = self.annotations[idx]
        targets = self.create_targets_tensor(bboxes)

        return image, targets

    def parse_annotation(self, annt):
      """
      Parse annotations for a single image and return list of [class_id, cx, cy, w, h]
      """
      parsed = []
      for obj in annt.get('objects', []):
        cname = obj.get('name')
        if cname not in self.classes:
          continue
        class_id = self.classes.index(cname)
        bnd = obj.get('bndbox')
        xmin, ymin, xmax, ymax = map(float, [bnd['xmin'], bnd['ymin'], bnd['xmax'], bnd['ymax']])
        w = xmax - xmin; h = ymax - ymin
        cx = xmin + w / 2.0; cy = ymin + h / 2.0
        parsed.append([class_id, np.round(cx), np.round(cy), np.round(w, 2), np.round(h, 2)])
      return parsed

    def create_targets_tensor(self, annotations):
        """
        Turn bbox annotations into learning targets
        param annotations: List of [class_id, x_center, y_center, width, height]
        """
        targets = torch.zeros((self.grid_height, self.grid_width, self.K + 5))
        for annotation in annotations:
            class_id, x_center, y_center, width, height = annotation

            cell_x_id = int(x_center // self.cell_width)
            cell_y_id = int(y_center // self.cell_height)

            if targets[cell_y_id, cell_x_id,  self.K+4] == 1:
              break # if there is already an object in the same grid

            targets[cell_y_id, cell_x_id, class_id] = 1  # Class score
            targets[cell_y_id, cell_x_id, self.K:self.K+2] = torch.tensor([
                2*(x_center - (cell_x_id + 1 - 0.5) * self.cell_width) / self.cell_width,
                2*(y_center - (cell_y_id + 1 - 0.5) * self.cell_height) / self.cell_height
            ])  # Normalized center coordinates

            targets[cell_y_id, cell_x_id,  self.K+2:self.K+4] = torch.tensor([
                width / self.img_width, height / self.img_height
            ])  # Normalized width and height

            targets[cell_y_id, cell_x_id,  self.K+4] = 1  # Confidence

        return targets

In [ ]:
 #### YOUR CODE STARTS HERE ####
from sklearn.model_selection import train_test_split
dataset_dir = globals().get('dataset_dir', 'dataset')
full_dataset = ObjectDetectionDataset(dataset_dir, classes, img_width, img_height, grid_height, grid_width)
indices = list(range(len(full_dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)
class SubsetDataset(Dataset):
    def __init__(self, base_dataset, indices):
        self.base = base_dataset
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        img = self.base.images[self.indices[idx]]
        ann = self.base.annotations[self.indices[idx]]
        img = self.base.transform(img)
        targets = self.base.create_targets_tensor(ann)
        return img, targets

train_dataset = SubsetDataset(full_dataset, train_idx)
val_dataset = SubsetDataset(full_dataset, val_idx)
#### YOUR CODE ENDS HERE ####

In [ ]:
len(train_dataset), len(val_dataset)

In [ ]:
def rel_to_abs_coord(bbox, grid_id_y, grid_id_x, img_width=img_width, img_height=img_height, cell_width=cell_width, cell_height=cell_height):
  """
  Convert grid-level coordinates into image-level coordinates
  """
  center_x = cell_width * grid_id_x + cell_width * (bbox[0]+ 1)/2
  center_y = cell_height * grid_id_y + cell_height * (bbox[1]+ 1)/2
  width = bbox[2] * img_width
  height = bbox[3] * img_height

  return (center_x, center_y, width, height)

def visualize_with_targets(ax, img, targets, num_classes, confidence_threshold = 0.5):
    """
    Visualize an image with bounding boxes on a given matplotlib axis.
    """
    if isinstance(img, torch.Tensor):
        img = img.permute(1, 2, 0).numpy()

    ax.imshow(img)
    ax.axis('off')

    if isinstance(targets, torch.Tensor):
        grid_height, grid_width = targets.size()[:2]
    else:
        grid_height, grid_width = targets.shape[:2]
    img_height, img_width = img.shape[:2]
    cell_width = img_width / grid_width
    cell_height = img_height / grid_height

    # Draw grid lines
    for x in range(1, grid_width):
        ax.axvline(x=cell_width * x, color='white', linestyle='-', linewidth=1)
    for y in range(1, grid_height):
        ax.axhline(y=cell_height * y, color='white', linestyle='-', linewidth=1)

    # Iterate over grid cells to draw bounding boxes
    for grid_id_y, grid_id_x in np.stack(np.where(targets[...,num_classes+4] >= confidence_threshold), axis = -1):

      # Convert center coordinates and sizes back into image coordinates
      center_x, center_y, width, height = rel_to_abs_coord(targets[grid_id_y, grid_id_x, num_classes:num_classes+4], grid_id_y, grid_id_x)
      top_left_x = center_x - (width / 2)
      top_left_y = center_y - (height / 2)

      colours = ['red', 'yellow', 'orange']
      class_id = torch.argmax(targets[grid_id_y, grid_id_x, :num_classes])
      # Filter out very small bboxes
      if (width / img_width) > 0.1 and (height / img_height) > 0.1:
          draw_rectangle(ax, top_left_x, top_left_y, width, height, colours[class_id])

          # adding centers to the image
          ax.scatter(center_x, center_y, color='red', s = 60)

In [ ]:
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

<font color='red'> Let's visualize a batch of images and targets from the DataLoader to make sure that they were loaded correctly.

In [ ]:
for image_batch, targets_batch in train_loader:
        fig, axes = plt.subplots(2, 4, figsize=(16, 16))
        axes = axes.flatten()

        for i in range(batch_size):
            ax = axes[i]
            img = image_batch[i]  # Current image tensor
            targets = targets_batch[i]  # Corresponding targets tensor

            # Call the visualization function for the current subplot axis
            visualize_with_targets(ax, img, targets, K, confidence_threshold = 0.35) # feel free to change the confidence threshold
        plt.show()
        break

<font color='red'>**(Homework exercise 1- c)** Train an object detection model.
In object detection practice session, we built a very simple model, using YOLO approach. Yet, the backbone of that model was too simple, therefore it failed to perform well on fruit dataset. Here, we ask you to replace the weak backbone with a more capable **pretrained classification model**, which you can find at [torchhub](https://pytorch.org/vision/stable/models.html). You need to replace its last layer with a suitable object detection head.
(2 points)</font>

In [ ]:
class AdvancedObjectDetector(nn.Module):
  def __init__(self, grid_height=4, grid_width=5, K=2, img_width = 416, img_height = 277):
    super(AdvancedObjectDetector, self).__init__()
    self.grid_height = grid_height
    self.grid_width = grid_width
    self.K = K

    # Pretrained backbone and a small detection head producing raw outputs
    import torchvision.models as models
    backbone = models.resnet18(pretrained=True)
    feat_dim = backbone.fc.in_features
    self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # remove final fc

    out_features = self.grid_height * self.grid_width * (self.K + 5)
    self.head = nn.Linear(feat_dim, out_features)

  def forward(self, x):
    # Extract features from backbone
    feats = self.backbone(x)  # [B, feat_dim, 1, 1]
    feats = feats.view(feats.size(0), -1)
    out = self.head(feats)
    return out

In [ ]:
# Initialize the model
obj_detection_model = AdvancedObjectDetector(grid_height=grid_height, grid_width=grid_width, K=K, img_width=img_width, img_height=img_height)
obj_detection_model = obj_detection_model.to(device)
obj_detection_model.train()  # Set the model to training mode

# Choose an optimizer
optimizer = torch.optim.Adam(obj_detection_model.parameters(), lr=0.0001)

In [ ]:
obj_detection_model

In [ ]:
classification_loss = nn.CrossEntropyLoss() # For classes
bbox_loss = nn.MSELoss()  # For bounding box coordinates and sizes
confidence_loss = nn.BCELoss()  # For object confidence scores

In [ ]:
def compute_loss(predictions, targets, lambda_coord=5, lambda_noobj=0.5, verbal=True):
    # Extract predictions
    pred_classes = predictions[..., :K]  # Assuming class scores are the first K channels
    pred_coords = predictions[..., K:K+2]  # Next two channels for coordinates
    pred_sizes = predictions[..., K+2:K+4]  # Sizes (width and height)
    pred_confidence = predictions[..., K+4]  # Confidence scores

    # Similar extraction needs to be done for targets based on our dataset structure
    # Extract targets
    targets_classes = targets[..., :K]
    targets_coords = targets[..., K:K+2]
    targets_sizes = targets[..., K+2:K+4]
    targets_confidence = targets[..., K+4]

    # Compute losses for each part

    loss_classes = classification_loss(pred_classes, targets_classes)
    loss_coords = bbox_loss(pred_coords, targets_coords)
    loss_sizes = bbox_loss(pred_sizes, targets_sizes)
    loss_confidence = confidence_loss(pred_confidence, targets_confidence)

    if verbal:
      print(f'Classes loss: {loss_classes}; Coordinates loss: {loss_coords}; Sizes loss: {loss_sizes}; Confidence loss: {loss_confidence}')

    # Combine losses
    total_loss = loss_coords + lambda_coord * loss_sizes + lambda_noobj * loss_confidence + loss_classes

    return total_loss

In [ ]:
def fit(model, loss_func, train_loader, val_loader, n_epochs):
  history = {'loss': [], 'val_loss': []}

  for epoch in range(n_epochs):
    # initialise losses for logging
    epoch_loss, val_epoch_loss = 0.0, 0.0

    model.train()
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()   # reseting gradients

        # Forward pass
        outputs = model(images)

        loss = loss_func(outputs, targets)

        # Backward and optimize
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    model.eval()
    with torch.inference_mode():
      for images, targets in val_loader:
          images, targets = images.to(device), targets.to(device)

          # Forward pass only
          outputs = model(images)
          loss = loss_func(outputs, targets)

          val_epoch_loss += loss.item()

    history['loss'].append(epoch_loss/len(train_loader))
    history['val_loss'].append(val_epoch_loss/len(val_loader))

    print(f"Epoch {epoch + 1}, Loss: {history['loss'][-1]}, Val loss: {history['val_loss'][-1]}")

  return history

In [ ]:
history = fit(obj_detection_model, compute_loss, train_loader, val_loader, 5) # feel free to change number of epochs

<font color='red'>**(Homework exercise 1- d)** Evaluate the object detection model. First, plot a batch of predictions for visual inspection and then calculate accuracy for object classes and IoU for bounding boxes. Getting a high IoU maybe challenging, so report the highest you could achieve. (1 point)


In [ ]:
# Set the model to evaluation mode
obj_detection_model.eval()

val_predictions = []

for image_batch, targets_batch in val_loader:
    with torch.inference_mode(): # disable gradient calculation
        # only feedforward pass
        val_preds = obj_detection_model(image_batch.to(device))
        val_preds = val_preds.detach().cpu()

        # save predictions to later be able to work with them
        val_predictions.append(val_preds.numpy())

        fig, axes = plt.subplots(2, 4, figsize=(16, 16))
        axes = axes.flatten()

        for i in range(batch_size):
            ax = axes[i]
            img = image_batch[i]  # Current image tensor
            targets = val_preds[i]  # Corresponding targets tensor

            # Call the visualization function for the current subplot axis
            visualize_with_targets(ax, img, targets, K, confidence_threshold = 0.35)
        plt.show()

<font color='red'> We talked about IoU in context of object segmentation. But it's also a very popular performance metric when it comes to object detection. Albeit, it's calculated in a slightly different way.

<font color='red'>To calculate IoU in object detection, it's crucial to first match each predicted bounding box with the corresponding ground truth bounding box. This matching process ensures that we are evaluating the model's predictions against the correct actual object locations.
This process goes as follows:</font>
1. Calculate IoU scores: for every ground thruth box, calculate the IoU with all predicted boxes.

2. Set IoU threshold: determine an IoU threshold (e.g., 0.5) that defines a successful match.

3. Match boxes: iteratively match each predicted box to the ground truth box with the highest IoU above the threshold, removing matched pairs from consideration.
4. Handle unmatched boxes: any predicted boxes without a match are false positives, and unmatched ground truth boxes are false negatives.
5. Calculate IoU and accuracy - at this point false positives are discarded.

<font color='red'> We provided you the evaluation code, you just need to implement IoU for bounding boxes function. You can test it by using ground thruth bounding boxes as predictions - you should get 1.

In [ ]:
def get_iou_bbox(ground_truth_box, predicted_box):
  """
  ground_truth_box: center_x, center_y, width, height
  predicted_box: center_x, center_y, width, height
  """
  #### YOUR CODE STARTS HERE ####
  gx, gy, gw, gh = ground_truth_box
  px, py, pw, ph = predicted_box

  g_xmin = gx - gw / 2.0; g_xmax = gx + gw / 2.0
  g_ymin = gy - gh / 2.0; g_ymax = gy + gh / 2.0

  p_xmin = px - pw / 2.0; p_xmax = px + pw / 2.0
  p_ymin = py - ph / 2.0; p_ymax = py + ph / 2.0

  inter_xmin = max(g_xmin, p_xmin)
  inter_ymin = max(g_ymin, p_ymin)
  inter_xmax = min(g_xmax, p_xmax)
  inter_ymax = min(g_ymax, p_ymax)

  inter_w = max(0.0, inter_xmax - inter_xmin)
  inter_h = max(0.0, inter_ymax - inter_ymin)
  inter_area = inter_w * inter_h

  g_area = (g_xmax - g_xmin) * (g_ymax - g_ymin)
  p_area = (p_xmax - p_xmin) * (p_ymax - p_ymin)

  union_area = g_area + p_area - inter_area
  iou = inter_area / union_area if union_area > 0 else 0.0
  #### YOUR CODE ENDS HERE ####

  return iou

In [ ]:
def evaluate_obj_detection(predictions, loader, K, confidence_threshold=0.35, iou_threshold=0.5):
  ious = []
  class_predictions = []

  for batch_indx, (images, targets) in enumerate(loader):
    targets = targets.numpy()
    for img_indx in range(targets.shape[0]):

      img_preds = np.copy(predictions[batch_indx][img_indx])

      # Find all grid cells with predicted objects in one image
      grid_ids = np.stack(np.where(img_preds[:, :, K+4] >= confidence_threshold), axis = -1)

      for grid_id_y, grid_id_x in np.stack(np.where(targets[img_indx, :, :,K+4] >= confidence_threshold), axis = -1):

        # Convert center coordinates and sizes back into image coordinates
        gt_bbox = rel_to_abs_coord(targets[img_indx, grid_id_y, grid_id_x, K:K + 4], grid_id_y, grid_id_x)

        # Calculate IoU with one ground truth bbox and all predicted bboxes
        gt_bbox_ious = [get_iou_bbox(gt_bbox, rel_to_abs_coord(img_preds[grid_id_y_p, grid_id_x_p, K:K + 4], grid_id_y, grid_id_x))
                        for grid_id_y_p, grid_id_x_p in grid_ids]
        gt_bbox_ious.append(0)

        max_iou = np.max(gt_bbox_ious)

        if max_iou > iou_threshold:
          ious.append(max_iou)

          # Find the matching predicted object based on the highest IoU
          object_grid_id_y, object_grid_id_x = grid_ids[np.argmax(gt_bbox_ious)]

          class_predictions.append(np.argmax(img_preds[object_grid_id_y, object_grid_id_x, :K], axis=0))

          # This object has already been assigned to the ground truth, remove it from predictions - not the most optimal approach, non maximum supression is usually used as a fist step
          img_preds[object_grid_id_y, object_grid_id_x, K+4] = 0
          grid_ids = np.stack(np.where(img_preds[:, :, K+4] >= confidence_threshold), axis = -1)

        else: # No objects were detected
          ious.append(0)
          class_predictions.append(K + 1) # Add non-existing class label - object wasn't classified

  return ious, class_predictions

In [ ]:
ious, object_classes = evaluate_obj_detection(val_predictions, val_loader, K, confidence_threshold=0.35) # feel free to change the confidence threshold
mean_iou = np.mean(ious)
acc = np.mean(object_classes == np.array([[a[0] for a in annt] for annt in val_dataset.annotations]).flatten())

print(f'The accuracy of object detector is {np.round(acc, 4)} and mean IoU is {np.round(mean_iou, 3)}')

# Homework exercise 2 (8 points): Create a segmentation model

<font color='red'>**(Homework exercise 2- a)** In this exercise, you will continue working with the dataset you collected for HW1. But this time we ask you to add segmentation masks annotations to as many images as you think will be enough to get a decent model. Choose **Semantic Segmentation with Polygons** option if you are using LabelStudio. Again, start with a small number of annotations (e.g., 5 per class) and then add more if the model's performance is too low (IoU < 0.9). (3 points)</font>

**LabelStudio notes**:
1. Create a new project and set the labeling setup to **Semantic Segmentation with Polygons**.
1. After you finished the labeling, download the images and annotations together as an archive.
2. Save annotations in the **COCO** format.

In [ ]:
# Upload your images and annotations archive
from google.colab import files

files.upload()

<font color='red'> Again, let's read a single image and the corresponding mask and visualize them to make sure that the mask annotations are correct.

In [ ]:
#### YOUR CODE STARTS HERE ####
import glob, os
dataset_dir = globals().get('dataset_dir', 'dataset')
# Try to find a mask corresponding to the first image
img_paths = glob.glob(os.path.join(dataset_dir, '**', '*.jpg'), recursive=True) + glob.glob(os.path.join(dataset_dir, '**', '*.png'), recursive=True)
if len(img_paths) == 0:
  raise FileNotFoundError('No images found for segmentation check. Update dataset_dir')
image_path = img_paths[0]
image = Image.open(image_path).convert('RGB')
# Try common mask locations: same dir with _mask, or masks/ folder
base = os.path.splitext(os.path.basename(image_path))[0]
mask_candidates = [os.path.join(os.path.dirname(image_path), base + '_mask.png'), os.path.join(os.path.dirname(image_path), base + '.png')]
mask_candidates += glob.glob(os.path.join(dataset_dir, 'masks', base + '*'))
mask = None
for m in mask_candidates:
  if os.path.exists(m):
    mask = Image.open(m).convert('L')
    break
if mask is None:
  # create an empty mask placeholder (all background) if none found
  mask = Image.new('L', image.size, 0)
#### YOUR CODE ENDS HERE ####

In [ ]:
plt.figure(figsize=(16,16))

plt.subplot(1, 2, 1)
plt.imshow(image, cmap='viridis')
plt.axis('off')
plt.title(f'Original Image')

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap='viridis')
plt.axis('off')
plt.title('Mask')
plt.show()

<font color='red'>**(Homework exercise 2- b)** Prepare segmentation dataset class. You can use `cv2.fillPoly` to get binary masks from a list of polygon vertices produced by LabelStudio. Since the segmentation model has to assign a class for every image pixel, you need to provide n masks for n classes + 1 mask for the background class. If you apply any geometric transformations to images (e.g., resizing), don't forget to do the same with masks. (2 points)

In [ ]:
class SegmentationDataset(Dataset):

    def __init__(self, dataset_dir, classes, img_width, img_height):
      self.dataset_dir = dataset_dir
      self.img_height = img_height
      self.img_width = img_width
      self.classes = classes
      self.K = len(classes)
      import glob, os, json, cv2
      # Try COCO-style annotations first
      ann_file = os.path.join(dataset_dir, 'annotations.json')
      images = []
      masks = []
      if os.path.exists(ann_file):
        with open(ann_file, 'r') as f:
          coco = json.load(f)
        # build image id -> filename mapping
        id2file = {img['id']: img['file_name'] for img in coco.get('images', [])}
        anns_by_image = {}
        for ann in coco.get('annotations', []):
          img_id = ann['image_id']
          anns_by_image.setdefault(img_id, []).append(ann)
        for img in coco.get('images', []):
          img_path = os.path.join(dataset_dir, img['file_name'])
          if not os.path.exists(img_path):
            continue
          images.append(Image.open(img_path).convert('RGB'))
          # create multi-channel mask
          h, w = img.get('height'), img.get('width')
          mask = np.zeros((h, w), dtype=np.uint8)
          for ann in anns_by_image.get(img['id'], []):
            # if polygons provided
            for poly in ann.get('segmentation', []):
              pts = np.array(poly).reshape(-1, 2).astype(np.int32)
              cv2.fillPoly(mask, [pts], color=1)
          masks.append(mask)
      else:
        # fallback: find mask files adjacent to images (e.g., base_mask.png)
        img_paths = glob.glob(os.path.join(dataset_dir, '**', '*.jpg'), recursive=True) + glob.glob(os.path.join(dataset_dir, '**', '*.png'), recursive=True)
        for ip in img_paths:
          base = os.path.splitext(os.path.basename(ip))[0]
          mask_candidates = [os.path.join(os.path.dirname(ip), base + '_mask.png'), os.path.join(os.path.dirname(ip), base + '.png')]
          mask_file = None
          for m in mask_candidates:
            if os.path.exists(m):
              mask_file = m; break
          if mask_file is None:
            continue
          images.append(Image.open(ip).convert('RGB'))
          masks.append(np.array(Image.open(mask_file).convert('L')))
      self.images = images
      self.masks = masks
      from torchvision import transforms as T
      self.image_transform = T.Compose([T.Resize((self.img_height, self.img_width)), T.ToTensor()])
      # mask_transform: convert single-channel mask to one-hot tensor (C x H x W)
      def mask_transform_fn(mask_arr):
        # mask_arr: H x W with integer labels (0 background, 1..K classes)
        h, w = mask_arr.shape
        one_hot = np.zeros((self.K + 1, h, w), dtype=np.uint8)
        for c in range(self.K + 1):
          one_hot[c] = (mask_arr == c).astype(np.uint8)
        return torch.from_numpy(one_hot.astype(np.float32))
      self.mask_transform = mask_transform_fn

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        mask = self.masks[idx]
        image = self.image_transform(image)
        mask = self.mask_transform(mask)
        return image, mask

    def parse_annotation(self, annt):
      # Parse a single annotation entry (COCO-like) into a binary mask
      import numpy as np, cv2
      h = annt.get('height', self.img_height)
      w = annt.get('width', self.img_width)
      mask = np.zeros((h, w), dtype=np.uint8)
      for obj in annt.get('objects', []):
        poly = np.array(obj.get('polygon')).reshape(-1, 2).astype(np.int32)
        cv2.fillPoly(mask, [poly], color=obj.get('label_id', 1))
      return mask

In [ ]:
#### YOUR CODE STARTS HERE ####
from sklearn.model_selection import train_test_split
dataset_dir = globals().get('dataset_dir', 'dataset')
full_seg_dataset = SegmentationDataset(dataset_dir, classes, img_width, img_height)
indices = list(range(len(full_seg_dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)
class SegSubset(Dataset):
  def __init__(self, base, idxs):
    self.base = base
    self.idxs = idxs
  def __len__(self):
    return len(self.idxs)
  def __getitem__(self, i):
    return self.base[i]
train_dataset = SegSubset(full_seg_dataset, train_idx)
val_dataset = SegSubset(full_seg_dataset, val_idx)
#### YOUR CODE ENDS HERE ####

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)

In [ ]:
def visualise_with_masks(image, mask, pred_mask=None):
  # Visualize an image and corresponding mask(s) side-by-side

  if isinstance(image, torch.Tensor):
    image = image.permute(1, 2, 0).numpy()
  if isinstance(mask, torch.Tensor):
    # Converts a one-hot encoded mask (C x H x W) to a single-channel mask (H x W X 1)
    # where each pixel's value corresponds to the class index with the highest probability.
    # Essentially, the channel with '1' (for each pixel) is mapped to its class index.
    mask = np.argmax(mask.permute(1, 2, 0).numpy(), axis=2)
  if isinstance(pred_mask, torch.Tensor):
    pred_mask = np.argmax(pred_mask.permute(1, 2, 0).numpy(), axis=2)

  n_plots = 3

  plt.figure(figsize=(16,16))
  norm = matplotlib.colors.Normalize(vmin=0, vmax=len(classes))

  plt.subplot(1, n_plots, 1)
  plt.imshow(image, cmap='viridis')
  plt.axis('off')
  plt.title(f'Original Image {image.shape}')

  plt.subplot(1, n_plots, 2)
  plt.imshow(mask, cmap='viridis', norm=norm)
  plt.axis('off')
  plt.title('Mask')

  if isinstance(pred_mask, np.ndarray):
    plt.subplot(1, n_plots, 3)
    plt.imshow(pred_mask, cmap='viridis', norm=norm)
    plt.axis('off')
    plt.title('Predicted Mask')

  plt.tight_layout()
  plt.show()

In [ ]:
for images, masks in train_loader:
  for img, mask in zip(images, masks):
    visualise_with_masks(img, mask)
    break

<font color='red'>**(Homework exercise 2- c)** Train a segmentation model.

There is no need to implement the U-net model from scratch and struggle with skip connections. We suggest you use the library with a very creative name [Segmentation Models](https://github.com/qubvel/segmentation_models.pytorch) which offers a variety of segmentation models and pre-trained backbones. (3 points)</font>

In [ ]:
! pip install segmentation-models-pytorch

In [ ]:
import segmentation_models_pytorch as smp

#### YOUR CODE STARTS HERE ####
# Use a pretrained encoder and set number of output classes = K + 1 (background)
segment_model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=len(classes) + 1)
#### YOUR CODE ENDS HERE ####
segment_model.train()
segment_model = segment_model.to(device)

In [ ]:
segmentation_loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(segment_model.parameters(), lr=0.001)

In [ ]:
history = fit(segment_model, segmentation_loss, train_loader, val_loader, 20) # feel free to change the number of epochs

<font color='red'>Now we will evaluate the segmentation model. Once again, we will calculate IoU score. </font>

In [ ]:
segment_model.eval()

predictions = []

for images, masks in val_loader:
  with torch.inference_mode():
    output = torch.nn.functional.softmax(segment_model(images.to(device)), dim=1) # Adding softmax activation to obtain class probabilities for each pixel
    predicted_masks = output.detach().cpu()
    predictions.append(predicted_masks.numpy())

  for img, mask, pred_mask in zip(images, masks, predicted_masks):
    visualise_with_masks(img, mask, pred_mask)

In [ ]:
def iou_numpy(gt_mask, pred_mask):
  """
  Calculate per channel IoU in a batch of masks
  """
  gt_mask = gt_mask.transpose(0,2,3,1).astype('bool')

  pred_mask = pred_mask.transpose(0,2,3,1)
  max_values = np.max(pred_mask, axis=-1, keepdims=True)
  pred_mask_binary = (pred_mask == max_values)

  intersection = (gt_mask & pred_mask_binary).sum((1,2))
  union = (gt_mask | pred_mask_binary).sum((1,2))

  iou = (intersection + 0.00001) / (union + 0.00001)

  return iou

In [ ]:
def evaluate_segmentation(predictions, loader):
  ious = []
  for (images, masks), pred_masks in zip(loader, predictions):
    masks = masks.numpy()
    ious.append(np.mean(iou_numpy(masks, pred_masks), axis=0))

  return np.mean(np.stack(ious), axis=0)

In [ ]:
ious = evaluate_segmentation(predictions, val_loader)
print(f'IoU for class {classes[0]}: {ious[1]}, for class {classes[1]}: {ious[2]}')

# Homework exercise 3 (4 points): Testing in the wild
<font color='red'>**(Homework exercise 3- a)** Now it's time to get your test set out of the safe you have been keeping it in and see how well the models will detect and segment objects 'in the wild'. Let's imagine that those images have been uploaded by the users of your object detection & segmentation service, so you don't have ground thruth bounding boxes and masks for them.
All you can do is to visually inspect model's predictions.


<font color='red'>It's a common practice to post-process segmentation masks produced  by the model to get even more refined segmentations. We suggest you use the predictions of object detection model for that purpose - leave only the object segmentations which fit inside the bounding boxes. There is no guarantee that it will lead to better results, so plot the predicted and refined masks side-by-side and decide for yourself. (3 points)

In [ ]:
# Upload the .zip archive with test images
from google.colab import files

files.upload()

In [ ]:
# No need to use dataset class or dataloader here
 #### YOUR CODE STARTS HERE ####
import glob, os
test_dir = globals().get('test_dir', 'test_images')
# find image files in test_dir or current dir
if os.path.exists(test_dir):
  image_paths = glob.glob(os.path.join(test_dir, '**', '*.jpg'), recursive=True) + glob.glob(os.path.join(test_dir, '**', '*.png'), recursive=True)
else:
  image_paths = glob.glob('*.jpg') + glob.glob('*.png')
from torchvision import transforms as T
image_transform = T.Compose([T.Resize((img_height, img_width)), T.ToTensor()])
#### YOUR CODE ENDS HERE ####

test_images = [image_transform(Image.open(img_path)) for img_path in image_paths]

In [ ]:
def postprocess_masks(bboxes, masks, confidence_threshold=0.35):
  # bboxes: grid_h x grid_w x (K+5) array (predictions for one image)
  # masks: torch Tensor (C, H, W) with per-class probabilities or logits
  import numpy as np
  grid_h, grid_w = bboxes.shape[:2]
  C, H, W = masks.shape
  # Create ROI mask where any predicted bbox (above threshold) covers
  roi = np.zeros((H, W), dtype=np.uint8)
  for gy in range(grid_h):
    for gx in range(grid_w):
      conf = bboxes[gy, gx, K+4]
      if conf >= confidence_threshold:
        cx, cy, w_rel, h_rel = bboxes[gy, gx, K:K+4]
        # convert relative (grid) coords to absolute using helper
        abs_cx, abs_cy, abs_w, abs_h = rel_to_abs_coord(bboxes[gy, gx, K:K+4], gy, gx)
        x1 = int(max(0, np.floor(abs_cx - abs_w / 2)))
        y1 = int(max(0, np.floor(abs_cy - abs_h / 2)))
        x2 = int(min(W, np.ceil(abs_cx + abs_w / 2)))
        y2 = int(min(H, np.ceil(abs_cy + abs_h / 2)))
        roi[y1:y2, x1:x2] = 1
  # Apply ROI to each class channel: zero out predictions outside ROIs
  refined = masks.clone().numpy() if isinstance(masks, torch.Tensor) else masks.copy()
  for c in range(refined.shape[0]):
    refined[c] = refined[c] * roi
  # Return refined masks in same format as input (tensor)
  refined_masks = torch.from_numpy(refined) if not isinstance(masks, torch.Tensor) else torch.from_numpy(refined).float()
  return refined_masks

In [ ]:
for img in test_images:
  img = img.unsqueeze(0).to(device)
  bboxes = obj_detection_model(img).detach().cpu().numpy()[0]
  pred_masks = torch.nn.functional.softmax(segment_model(img), dim=1).detach().cpu()[0]

  refined_masks = postprocess_masks(bboxes, pred_masks)

  visualise_with_masks(img.cpu()[0], pred_masks, refined_masks)

<font color='red'>**(Homework exercise 3- b)** What can you say about the test results? (1 point)

Your answer:

# Comments (optional feedback to the course instructors)
Here, please, leave your comments regarding the homework, possibly answering the following questions:
* how much time did you send on this homework?
* was it too hard/easy for you?
* what would you suggest to add or remove?
* anything else you would like to tell us

Your comments:

# <font color='red'>  End of the homework. Please don't delete this cell.</font>